# EDA -- silver

What columns each table in `data/silver/` has and how they relate to each other. No full dumps -- maximum 5 sample rows per table.

In [1]:
import os
from pathlib import Path

import duckdb
import pandas as pd


def _repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise FileNotFoundError("pyproject.toml not found above " + str(start))


os.chdir(_repo_root(Path.cwd()))

AIRE_PATH = "data/silver/aire.parquet"
ESTACIONES_AIRE_PATH = "data/silver/estaciones_aire.parquet"
TRAFICO_PATH = "data/silver/trafico.parquet"

## `aire`

Bronze -> silver (`unpivot_air_quality`, `src/data/silver/aire.py`):

- Wide -> long pivot: columns `D01..D31`/`V01..V31` (data/validity per day) become rows `fecha`/`dato`/`validez` (`aire.py:33-44`).
- `MAGNITUD` (numeric code, e.g. `8`) -> `magnitud` (label, e.g. `"NO2"`) via the `MAGNITUD_LABELS` dict (`aire.py:15-19,29-31`).
- Days that don't exist in that month's calendar are discarded -- bronze fills them with `dato=0.0`/`validez='N'` instead of NULL, so the filter is on `LAST_DAY`, not nullity (`aire.py:23-28,39-41`).
- Dropped columns: `PROVINCIA`, `MUNICIPIO`, `PUNTO_MUESTREO`, `ANO`, `MES`.

In [2]:
aire = pd.read_parquet(AIRE_PATH)
aire.shape

(157741, 5)

In [3]:
aire.dtypes

estacion      int64
magnitud     object
fecha        object
dato        float64
validez      object
dtype: object

In [4]:
aire.head(5)

,estacion,magnitud,fecha,dato,validez
0,11,NOx,2025-06-30,24.0,V
1,11,TOL,2025-03-30,0.5,V
2,11,TOL,2026-03-30,0.7,V
3,11,BEN,2026-04-30,0.2,V
4,11,EBE,2025-03-30,0.1,V


In [5]:
aire.isna().sum()

estacion    0
magnitud    0
fecha       0
dato        0
validez     0
dtype: int64

## `estaciones_aire`

Bronze -> silver (`assign_district`, `src/data/silver/district_join.py`): same columns as bronze plus `COD_DIS` and `NOMBRE`, added by a spatial join (`gpd.sjoin(..., predicate="within")`) between `Point(LONGITUD, LATITUD)` of each station and the geometry of `distritos`. No other column changes.

In [6]:
estaciones_aire = pd.read_parquet(ESTACIONES_AIRE_PATH)
estaciones_aire.shape

(24, 27)

In [7]:
estaciones_aire.dtypes

CODIGO                          int64
CODIGO_CORTO                    int64
ESTACION                       object
DIRECCION                      object
LONGITUD_ETRS89                object
LATITUD_ETRS89                 object
ALTITUD                         int64
COD_TIPO                       object
NOM_TIPO                       object
NO2                            object
SO2                            object
CO                             object
PM10                           object
PM2_5                          object
O3                             object
BTX                            object
COD_VIA                         int64
VIA_CLASE                      object
VIA_PAR                        object
VIA_NOMBRE                     object
Fecha alta             datetime64[us]
COORDENADA_X_ETRS89            object
COORDENADA_Y_ETRS89            object
LONGITUD                      float64
LATITUD                       float64
COD_DIS                        object
NOMBRE      

In [8]:
estaciones_aire.head(5)

,CODIGO,CODIGO_CORTO,ESTACION,DIRECCION,LONGITUD_ETRS89,LATITUD_ETRS89,ALTITUD,COD_TIPO,NOM_TIPO,NO2,...,VIA_CLASE,VIA_PAR,VIA_NOMBRE,Fecha alta,COORDENADA_X_ETRS89,COORDENADA_Y_ETRS89,LONGITUD,LATITUD,COD_DIS,NOMBRE
0,28079004,4,Plaza de España,Plaza de España,"3°42'43.91""O","40°25'25.98""N",637,UT,Urbana tráfico,X,...,PLAZA,DE,ESPAÑA,1998-12-01,"439579,3291","4475049,263",-3.712257,40.423882,9,Moncloa - Aravaca
1,28079008,8,Escuelas Aguirre,Entre C/ Alcalá y C/ O’ Donell,"3°40'56.22""O","40°25'17.63""N",672,UT,Urbana tráfico,X,...,CALLE,DE,ALCALA,1998-12-01,"442117,2366","4474770,696",-3.682316,40.421553,4,Salamanca
2,28079011,11,Ramón y Cajal,Avda. Ramón y Cajal esq. C/ Príncipe de Vergara,"3°40'38.50""O","40°27'5.29""N",709,UT,Urbana tráfico,X,...,CALLE,DEL,PRINCIPE DE VERGARA,1998-12-01,"442564,0457","4478088,595",-3.677349,40.451473,5,Chamartín
3,28079016,16,Arturo Soria,C/ Arturo Soria esq. C/ Vizconde de los Asilos,"3°38'21.17""O","40°26'24.20""N",695,UF,Urbana fondo,X,...,CALLE,DEL,VIZCONDE DE LOS ASILOS,1998-12-01,"445786,1729","4476796,019",-3.639242,40.440046,15,Ciudad Lineal
4,28079017,17,Villaverde,C/ Juan Peñalver,"3°42'47.89""O","40°20'49.74""N",601,UF,Urbana fondo,X,...,CALLE,DE,JUAN PEÑALVER,1998-12-01,"439420,7015","4466532,455",-3.713317,40.347147,17,Villaverde


In [9]:
estaciones_aire.isna().sum()

CODIGO                  0
CODIGO_CORTO            0
ESTACION                0
DIRECCION               0
LONGITUD_ETRS89         0
LATITUD_ETRS89          0
ALTITUD                 0
COD_TIPO                0
NOM_TIPO                0
NO2                     0
SO2                    20
CO                     20
PM10                   11
PM2_5                  16
O3                     11
BTX                    19
COD_VIA                 0
VIA_CLASE               1
VIA_PAR                 2
VIA_NOMBRE              1
Fecha alta              0
COORDENADA_X_ETRS89     0
COORDENADA_Y_ETRS89     0
LONGITUD                0
LATITUD                 0
COD_DIS                 0
NOMBRE                  0
dtype: int64

## `trafico`

Tabla grande -- se consulta con DuckDB directamente sobre el parquet, sin cargarla entera en memoria.

Bronze -> silver (`clean_traffic`, `src/data/silver/trafico.py`): same schema, column by column, as bronze -- only value-level change: negative sentinel -> `NULL` in `intensidad`/`ocupacion`/`carga`/`vmed` (`trafico.py:18-27`), because negative means "no data" per official CKAN documentation.

In [10]:
duckdb.sql(f"SELECT count(*) AS n_filas FROM '{TRAFICO_PATH}'")

┌──────────┐
│ n_filas  │
│  int64   │
├──────────┤
│ 89431963 │
└──────────┘

In [11]:
duckdb.sql(f"DESCRIBE SELECT * FROM '{TRAFICO_PATH}'")

┌─────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name     │ column_type │  null   │   key   │ default │  extra  │
│       varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id                  │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ fecha               │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ tipo_elem           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ intensidad          │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ ocupacion           │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ carga               │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ vmed                │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ error               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ periodo_integracion │ BIGINT      │ YES     │ NULL    │ NULL  

In [12]:
duckdb.sql(f"SELECT * FROM '{TRAFICO_PATH}' LIMIT 5")

┌───────┬─────────────────────┬───────────┬────────────┬───────────┬────────┬────────┬─────────┬─────────────────────┐
│  id   │        fecha        │ tipo_elem │ intensidad │ ocupacion │ carga  │  vmed  │  error  │ periodo_integracion │
│ int64 │      timestamp      │  varchar  │   double   │  double   │ double │ double │ varchar │        int64        │
├───────┼─────────────────────┼───────────┼────────────┼───────────┼────────┼────────┼─────────┼─────────────────────┤
│  1047 │ 2023-01-08 23:45:00 │ C30       │      540.0 │       2.0 │    0.0 │   53.0 │ N       │                  10 │
│  1047 │ 2023-01-09 04:00:00 │ C30       │      540.0 │       2.0 │    0.0 │   53.0 │ N       │                  10 │
│  1047 │ 2023-01-14 10:15:00 │ C30       │      432.0 │       2.0 │    0.0 │   61.0 │ N       │                  10 │
│  1047 │ 2023-01-20 10:30:00 │ C30       │      852.0 │       4.0 │    0.0 │   61.0 │ N       │                  10 │
│  1047 │ 2023-01-23 08:00:00 │ C30       │     

In [13]:
duckdb.sql(f"""
    SELECT
        count(*) FILTER (WHERE intensidad IS NULL) AS null_intensidad,
        count(*) FILTER (WHERE ocupacion IS NULL) AS null_ocupacion,
        count(*) FILTER (WHERE carga IS NULL) AS null_carga,
        count(*) FILTER (WHERE vmed IS NULL) AS null_vmed
    FROM '{TRAFICO_PATH}'
""")

┌─────────────────┬────────────────┬────────────┬───────────┐
│ null_intensidad │ null_ocupacion │ null_carga │ null_vmed │
│      int64      │     int64      │   int64    │   int64   │
├─────────────────┼────────────────┼────────────┼───────────┤
│               0 │              0 │          0 │         0 │
└─────────────────┴────────────────┴────────────┴───────────┘

## Relationships between tables

- `aire.estacion` <-> `estaciones_aire` (station key): join necessary to have coordinates/district for each air reading.
- `trafico` needs no join with anything: its metadata already carries its own `distrito` (documented in phase 1 -- `spec/features/001-descubrimiento-catalago/findings.md`).

## About `object` dtypes

Many columns come out as `object` in pandas. Not all for the same reason:

1. **Genuine strings** (`magnitud`, `validez`, `tipo_elem`, `error`, `COD_TIPO`, `COD_DIS`...): `object` is correct, not a bug. Candidates for `.astype("category")` if memory optimization is desired (low cardinality), but not urgent.
2. **`aire.fecha` is the odd case**: in the parquet it's a native `DATE` (duckdb confirms it), but `pd.read_parquet` materializes it as `object` (Python `datetime.date` objects) instead of `datetime64[ns]`, because pyarrow converts a `date32` to `object` by default unless explicitly requested via `pd.to_datetime(...)`. Compare with `trafico.fecha` (already `datetime64[us]`, because bronze carries it as timestamp) and with `estaciones_aire["Fecha alta"]` (same `DATE` in bronze, but comes out `datetime64[us]` because that path goes through `duckdb.sql(...).df()` instead of `pd.read_parquet` directly). Recommendation if treating as real date in pandas/Power BI/Gradio: `pd.to_datetime(aire["fecha"])` after load.
3. **Numeric columns never parsed**: `estaciones_aire.COORDENADA_X_ETRS89`/`_Y_ETRS89` (string with comma decimal, e.g. `"439579,3291"`) and `LONGITUD_ETRS89`/`LATITUD_ETRS89` (string in DMS format, e.g. `3°42'43.91"O`) -- redundant with already-clean columns `LONGITUD`/`LATITUD` (float), so no urgency to fix unless used directly.

In [14]:
aire.dtypes

estacion      int64
magnitud     object
fecha        object
dato        float64
validez      object
dtype: object